# Mouse annotator comparison: IgBLAST vs abstar (200 reads)

Сравнение уже готовых smoke-аннотаций для мыши `ERP003950`, sample `ERR346596`, 200 merged sequences.

Оба аннотатора запускались с mouse-compatible germline DB:
- IgBLAST — `/data/user/epishkin/igblast/internal_data/mouse/mouse_gl_V/D/J`
- abstar — `c57bl6`

При каждом запуске notebook заново читает текущие TSV и перезаписывает `comparison_report_mouse.json`.


In [ ]:
# CELL 1: Setup
import csv, json
from pathlib import Path
from collections import Counter

DATASET = "ERP003950"
SAMPLE = "ERR346596"
N = 200
BASE = Path("/data/user/epishkin/results") / DATASET
COMPARE = BASE / "annotator_compare"

IG_TSV = COMPARE / "output" / "igblast" / f"{SAMPLE}_{N}_igblast.tsv"
AB_TSV = COMPARE / "output" / "abstar" / SAMPLE / "airr" / f"{SAMPLE}_{N}.tsv"
REPORT = COMPARE / "comparison_report_mouse.json"

print("DATASET:", DATASET)
print("SAMPLE:", SAMPLE)
print("IgBLAST:", IG_TSV, "exists=", IG_TSV.exists(), "size=", IG_TSV.stat().st_size if IG_TSV.exists() else None)
print("abstar: ", AB_TSV, "exists=", AB_TSV.exists(), "size=", AB_TSV.stat().st_size if AB_TSV.exists() else None)


In [ ]:
# CELL 2: If needed, normalize existing outputs into canonical annotator_compare layout
# This cell does not rerun annotation. It only fixes path layout if an older flat local copy exists.

flat_ab = COMPARE / "output" / "abstar" / f"{SAMPLE}_{N}.tsv"
AB_TSV.parent.mkdir(parents=True, exist_ok=True)

if AB_TSV.exists() and AB_TSV.stat().st_size > 0:
    print("SKIP exists:", AB_TSV)
elif flat_ab.exists() and flat_ab.stat().st_size > 0:
    AB_TSV.write_bytes(flat_ab.read_bytes())
    print("COPIED:", flat_ab, "->", AB_TSV)
else:
    print("No flat abstar copy to normalize")

print("
Final files:")
for p in [IG_TSV, AB_TSV]:
    print(p, "exists=", p.exists(), "size=", p.stat().st_size if p.exists() else None)


In [ ]:
# CELL 3: Load TSVs and check shared sequence IDs

def load_tsv(path):
    rows = {}
    with open(path, newline="") as f:
        reader = csv.DictReader(f, delimiter="	")
        for row in reader:
            rows[row["sequence_id"]] = row
    return rows

if not IG_TSV.exists():
    raise FileNotFoundError(f"IgBLAST TSV missing: {IG_TSV}")
if not AB_TSV.exists():
    raise FileNotFoundError(f"abstar TSV missing: {AB_TSV}")

igrows = load_tsv(IG_TSV)
abrows = load_tsv(AB_TSV)
shared = sorted(set(igrows) & set(abrows))

print(f"IgBLAST reads: {len(igrows)}")
print(f"abstar reads:  {len(abrows)}")
print(f"Shared reads:  {len(shared)}")
print("First 5 shared IDs:", shared[:5])


In [ ]:
# CELL 4: Compare productivity, V/D/J calls, and CDR3 AA

def is_productive(row):
    return row.get("productive", "").strip().lower() in {"t", "true", "1", "yes"}

def pct(a, b):
    return round((a / b * 100), 1) if b else 0.0

ig_prod = sum(is_productive(r) for r in igrows.values())
ab_prod = sum(is_productive(r) for r in abrows.values())

stats = {
    "dataset": DATASET,
    "sample": SAMPLE,
    "reads_requested": N,
    "igblast": {"total": len(igrows), "productive": ig_prod, "productive_pct": pct(ig_prod, len(igrows)), "germline_db": "mouse_gl"},
    "abstar": {"total": len(abrows), "productive": ab_prod, "productive_pct": pct(ab_prod, len(abrows)), "germline_db": "c57bl6"},
    "shared_reads": len(shared),
}

for field in ["v_call", "d_call", "j_call", "cdr3_aa"]:
    agree = sum(igrows[s].get(field, "").strip() == abrows[s].get(field, "").strip() for s in shared)
    stats[field + "_agree"] = agree
    stats[field + "_agree_pct"] = pct(agree, len(shared))

print("Productive IgBLAST:", f"{ig_prod}/{len(igrows)}", f"({stats['igblast']['productive_pct']}%)")
print("Productive abstar: ", f"{ab_prod}/{len(abrows)}", f"({stats['abstar']['productive_pct']}%)")
print()
print(f"On {len(shared)} shared reads:")
for field in ["v_call", "d_call", "j_call", "cdr3_aa"]:
    print(f"  {field:8s}: {stats[field + '_agree']}/{len(shared)} ({stats[field + '_agree_pct']}%)")


In [ ]:
# CELL 5: Show call distributions and first examples

def top_counts(rows, field, n=10):
    c = Counter((r.get(field, "") or "<EMPTY>").strip() or "<EMPTY>" for r in rows.values())
    return c.most_common(n)

print("Top IgBLAST V calls:")
for k, v in top_counts(igrows, "v_call"):
    print(f"  {v:3d}  {k}")

print("
Top abstar V calls:")
for k, v in top_counts(abrows, "v_call"):
    print(f"  {v:3d}  {k}")

print("
First 10 shared reads: IgBLAST -> abstar")
for sid in shared[:10]:
    print(f"{sid}")
    print("  V:", igrows[sid].get("v_call", ""), "->", abrows[sid].get("v_call", ""))
    print("  J:", igrows[sid].get("j_call", ""), "->", abrows[sid].get("j_call", ""))
    print("  CDR3_AA:", igrows[sid].get("cdr3_aa", ""), "->", abrows[sid].get("cdr3_aa", ""))


In [ ]:
# CELL 6: Save JSON report
report = {
    **stats,
    "notes": [
        "Mouse comparison uses mouse-compatible germline DBs in both annotators.",
        "IgBLAST and abstar use different germline database nomenclatures, so V/D/J exact string agreement can be low even when annotations are biologically similar.",
        "CDR3_AA is the most portable direct comparison field across these annotators."
    ],
}
REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("WROTE:", REPORT)
print(json.dumps(report, indent=2, ensure_ascii=False))
